# VPU01 depression-storage validation (issue #38)

Sanity-check outputs from the `feat/depstor-validation-issue-38` cluster run.
Reads the 7 depstor rasters and 4 zonal-stat CSV groups under `gfv2_vpu01/` and confirms:
- Rasters are non-empty, correct dtypes, value distributions in the expected band.
- Zonal CSVs have one row per VPU01 HRU (~11,278), no NaNs, fractions in [0, 1].

In [ ]:
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling

DATA_ROOT = Path('/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2_param_v2')
FABRIC_DIR = DATA_ROOT / 'gfv2_vpu01'
RASTER_DIR = FABRIC_DIR / 'depstor_rasters'
PARAMS_DIR = FABRIC_DIR / 'params'
EXPECTED_HRU_COUNT = 11278

## Raster summaries

In [ ]:
RASTERS = [
    ('imperv_binary.tif', 'imperv >= 50%'),
    ('stream_buffer.tif', '60m stream buffer'),
    ('wbody_binary.tif', 'waterbodies'),
    ('wbody_regions.tif', 'connected-component IDs'),
    ('dprst_binary.tif', 'true depression storage'),
    ('onstream_binary.tif', 'on-stream water bodies'),
    ('drains_to_dprst.tif', 'cells draining to dprst'),
]

rows = []
for fname, label in RASTERS:
    p = RASTER_DIR / fname
    if not p.exists():
        rows.append({'raster': fname, 'status': 'MISSING'})
        continue
    with rasterio.open(p) as ds:
        arr = ds.read(1)
        nodata = ds.nodata
        valid = arr if nodata is None else arr[arr != nodata]
        row = {
            'raster': fname,
            'label': label,
            'shape': f'{ds.height}x{ds.width}',
            'dtype': str(arr.dtype),
            'nodata': nodata,
            'valid_pct': round(100 * valid.size / arr.size, 2),
            'min': float(valid.min()) if valid.size else None,
            'max': float(valid.max()) if valid.size else None,
            'mean': float(valid.mean()) if valid.size else None,
        }
        if arr.dtype == np.uint8 and ds.nodata == 255:
            row['frac_ones'] = round(100 * (arr == 1).sum() / arr.size, 3)
        if 'regions' in fname:
            row['n_regions'] = int(valid.max()) if valid.size else 0
        rows.append(row)

pd.DataFrame(rows)

## Thumbnail grid (decimated)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for ax, (fname, label) in zip(axes.flat, RASTERS):
    p = RASTER_DIR / fname
    if not p.exists():
        ax.set_title(f'{fname}\n(missing)')
        ax.axis('off')
        continue
    with rasterio.open(p) as ds:
        out_shape = (1, max(1, ds.height // 50), max(1, ds.width // 50))
        thumb = ds.read(out_shape=out_shape, resampling=Resampling.nearest)[0]
    masked = np.ma.masked_equal(thumb, 255) if thumb.dtype == np.uint8 else thumb
    ax.imshow(masked, cmap='viridis', interpolation='nearest')
    ax.set_title(f'{fname}\n{label}')
    ax.axis('off')
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()

## Zonal CSV sanity checks

In [ ]:
csv_rows = []
for src in ['dprst_frac', 'imperv_frac', 'onstream_storage_frac', 'drains_to_dprst_frac']:
    paths = sorted(glob.glob(str(PARAMS_DIR / src / '*.csv')))
    if not paths:
        csv_rows.append({'source': src, 'status': 'MISSING'})
        continue
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    val_col = [c for c in df.columns if c != 'nat_hru_id'][0]
    vals = df[val_col]
    csv_rows.append({
        'source': src,
        'batch_files': len(paths),
        'rows': len(df),
        'unique_hrus': df['nat_hru_id'].nunique(),
        'nans': int(vals.isna().sum()),
        'min': float(vals.min(skipna=True)),
        'max': float(vals.max(skipna=True)),
        'mean': float(vals.mean(skipna=True)),
        'frac_zero': round(100 * (vals == 0).sum() / len(vals), 2),
        'in_range': bool(vals.between(0, 1).all()),
        'count_match': df['nat_hru_id'].nunique() == EXPECTED_HRU_COUNT,
    })

pd.DataFrame(csv_rows)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, src in zip(axes, ['dprst_frac', 'imperv_frac', 'onstream_storage_frac', 'drains_to_dprst_frac']):
    paths = sorted(glob.glob(str(PARAMS_DIR / src / '*.csv')))
    if not paths:
        ax.set_title(f'{src}\n(missing)')
        continue
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    val_col = [c for c in df.columns if c != 'nat_hru_id'][0]
    ax.hist(df[val_col].dropna(), bins=50)
    ax.set_yscale('log')
    ax.set_title(f'{src}\n(n={len(df)})')
    ax.set_xlabel('fraction')
plt.tight_layout()
plt.show()